In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Function to generate a new dataset
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [4]:
# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [5]:
# Passes a test case into Claude
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [6]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score
                }

In [7]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [8]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [9]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere are several solutions to parse an S3 bucket name from a full S3 URI:\n\n## Solution 1: Using String Methods (Simple)\n\n```python\ndef parse_s3_bucket(s3_uri):\n    \"\"\"Extract bucket name from S3 URI\"\"\"\n    # Remove s3:// prefix and split by /\n    bucket = s3_uri.replace(\"s3://\", \"\").split(\"/\")[0]\n    return bucket\n\n# Test\nprint(parse_s3_bucket(\"s3://my-bucket/path/to/file.txt\"))  # Output: my-bucket\nprint(parse_s3_bucket(\"s3://another-bucket/\"))  # Output: another-bucket\n```\n\n## Solution 2: Using Regular Expressions\n\n```python\nimport re\n\ndef parse_s3_bucket_regex(s3_uri):\n    \"\"\"Extract bucket name using regex\"\"\"\n    match = re.match(r\"s3://([^/]+)\", s3_uri)\n    return match.group(1) if match else None\n\n# Test\nprint(parse_s3_bucket_regex(\"s3://my-bucket/path/to/file.txt\"))  # Output: my-bucket\nprint(parse_s3_bucket_regex(\"invalid-uri\"))  # Output: None\n```\n\n## Solution 3: Usin